In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "processed" / "nepal_wdi_2010_2024_working.csv"
DICTIONARY_FILE = ROOT / "data" / "metadata" / "variable_dictionary.csv"

df = pd.read_csv(DATA_FILE)
dictionary = pd.read_csv(DICTIONARY_FILE)

indicator_columns = dictionary["Indicator"].tolist()

print("Dataset loaded:", DATA_FILE.resolve())
print("Rows:", len(df))
print("Indicators:", len(indicator_columns))

display(df.head())

Dataset loaded: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/data/processed/nepal_wdi_2010_2024_working.csv
Rows: 15
Indicators: 9


,Year,Country,Country_Code,GDP_per_capita,GDP_growth,Life_expectancy,Basic_drinking_water,Secondary_school_enrollment,Electricity_access,Internet_users,Forest_area,CO2_per_capita_excl_LULUCF
0,2010,Nepal,NPL,721.265221,4.816415,66.772,85.761759,55.589291,68.6,7.9300,41.590722,0.175006
1,2011,Nepal,NPL,742.617253,3.421809,67.123,86.359902,58.993511,67.3,9.0000,41.590722,0.195187
2,2012,Nepal,NPL,775.310578,4.670142,67.364,86.953284,63.141479,75.2,11.1493,41.590722,0.226080
3,2013,Nepal,NPL,801.039061,3.525153,67.598,87.541580,65.493530,77.4,13.3000,41.590722,0.228210
4,2014,Nepal,NPL,846.665542,6.011483,67.812,88.124806,67.531830,84.9,15.4400,41.590722,0.265695


In [2]:
values = df[indicator_columns]

q1 = values.quantile(0.25, interpolation="linear")
q3 = values.quantile(0.75, interpolation="linear")

descriptive = pd.DataFrame({
    "N": values.count(),
    "Mean": values.mean(),
    "Median": values.median(),
    "Min": values.min(),
    "Max": values.max(),
    "Range": values.max() - values.min(),
    "Variance": values.var(ddof=1),
    "SD": values.std(ddof=1),
    "Q1": q1,
    "Q3": q3,
    "IQR": q3 - q1,
})

descriptive.index.name = "Indicator"
descriptive = descriptive.reset_index()

unit_lookup = dictionary.set_index("Indicator")["Unit"]
descriptive.insert(
    1, "Unit", descriptive["Indicator"].map(unit_lookup)
)

TABLE_FILE = ROOT / "results" / "tables" / "descriptive_statistics.csv"
TABLE_FILE.parent.mkdir(parents=True, exist_ok=True)

# Save full precision; round only the displayed table.
descriptive.to_csv(TABLE_FILE, index=False)

display(descriptive.round(4))
print("Saved:", TABLE_FILE.resolve())

,Indicator,Unit,N,Mean,Median,Min,Max,Range,Variance,SD,Q1,Q3,IQR
0,GDP_per_capita,constant 2015 US$ per person,15,947.5189,951.8573,721.2652,1179.9550,458.6898,23203.3875,152.3266,823.8523,1069.9534,246.1011
1,GDP_growth,annual %,15,4.2581,4.6701,-2.3696,8.9773,11.3469,7.8936,2.8096,3.4735,5.8214,2.3479
2,Life_expectancy,years,15,68.5420,68.4440,66.7720,70.6360,3.8640,1.4704,1.2126,67.4860,69.2025,1.7165
3,Basic_drinking_water,% of population,15,89.7884,89.8421,85.7618,93.6401,7.8784,6.3453,2.5190,87.8332,91.7791,3.9459
4,Secondary_school_enrollment,% gross,14,74.2458,73.0929,55.5893,90.3878,34.7986,132.1898,11.4974,66.0031,84.0575,18.0544
5,Electricity_access,% of population,15,85.3000,89.9000,67.3000,97.9000,30.6000,86.9357,9.3239,79.7000,91.0000,11.3000
6,Internet_users,% of population,15,26.4581,24.3000,7.9300,46.3219,38.3919,199.7638,14.1338,14.3700,38.9772,24.6072
7,CO2_per_capita_excl_LULUCF,t CO2e/capita,15,0.3957,0.4712,0.1750,0.5898,0.4147,0.0232,0.1524,0.2457,0.5256,0.2799
8,Forest_area,% of land area,14,41.5907,41.5907,41.5907,41.5907,0.0000,0.0000,0.0000,41.5907,41.5907,0.0000


Saved: /Users/zsres/Documents/M.Tech/Statistical-Inference-Assignment/results/tables/descriptive_statistics.csv


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = (
    ROOT / "data" / "processed"
    / "nepal_wdi_2010_2024_working.csv"
)

nepal = pd.read_csv(DATA_FILE).sort_values("Year").reset_index(drop=True)

TABLE_DIR = ROOT / "results" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Loaded from:", DATA_FILE.resolve())
print("Rows, columns:", nepal.shape)

display(nepal.head())

In [ ]:
indicator_columns = [
    "GDP_per_capita",
    "GDP_growth",
    "Life_expectancy",
    "Basic_drinking_water",
    "Secondary_school_enrollment",
    "Electricity_access",
    "Internet_users",
    "CO2_per_capita_excl_LULUCF",
    "Forest_area",
]

units = {
    "GDP_per_capita": "constant 2015 US$ per person",
    "GDP_growth": "annual %",
    "Life_expectancy": "years",
    "Basic_drinking_water": "% of population",
    "Secondary_school_enrollment": "% gross",
    "Electricity_access": "% of population",
    "Internet_users": "% of population",
    "CO2_per_capita_excl_LULUCF": "t CO2e/capita; excluding LULUCF",
    "Forest_area": "% of land area",
}

data = nepal[indicator_columns]

display(data.count().rename("Available_N").to_frame())